# CODE 3.2 — Learn and train MiniBatch K-Means
**IOTG-62 · G3B / N-BaIoT · Beginner notebook**

Read `START_HERE.md` for installation. Run each cell in order with **Shift+Enter**.
This notebook contains explanations and executable training code, rather than precomputed results.
The actual G3B package is required; the handoff PDF is a description of that package.

**Workflow:** notebook 01 (pilot) → notebook 02 (pilot) → change the JSON stage to `full`
→ notebook 01 (full) → notebook 02 (full). Restart the kernel before each run.

The pilot is a declared 100,000-row TRAIN sample. Finalists then see **all 4,944,968 TRAIN rows**
in bounded batches. Neither stage fits on VALIDATION or TEST. Labels never enter fitting.


## 1. What are we asking the algorithm to learn?
In classification, an example is `(115 features, benign/malicious label)` and the label teaches
the model the desired answer. Here the model receives **only the 115 features** and groups
records that are close together.

| Term | Meaning here |
|---|---|
| Record / row | One observation in G3B |
| Feature / column | One numeric predictor; keep G3B's canonical order |
| Cluster | Records assigned to the same learned center |
| Centroid | A learned center in the 115-dimensional feature space |
| K | How many centers we ask the model to learn |
| Ground truth `y` | Known benign/malicious labels, reserved for later analysis |

K-Means repeatedly assigns rows to the nearest center and updates centers. Its objective is
\(\sum_i \|x_i-\mu_{c_i}\|^2\): make each row close to its assigned center.
MiniBatch K-Means approximates these updates with small batches. It is suitable for this package's
millions of rows; it does not need a GPU.

**Cluster IDs have no class meaning.** Cluster 0 does not automatically mean benign.
K=2 asks for two geometric groups; it does not teach the model two security classes.


## 2. Configuration and imports
Edit `experiment_config.json`, rather than changing values in many cells. At first use `stage: pilot`.
`random_state` controls repeatable random choices. `threads` limits CPU parallelism.
A fixed seed improves repeatability, but retraining across different library/hardware versions can
still differ; the environment and model are saved.


In [1]:
import json
import os
from pathlib import Path

CONFIG_FILE = Path("experiment_config.json")  # Can point to an archived configuration for retraining.
initial_config = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = str(initial_config["threads"])

import itertools
import random
import shutil
import time
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, MiniBatchKMeans, kmeans_plusplus
from threadpoolctl import threadpool_limits

from clustering_utils import (
    comparison_policy, open_package, prepare_cache, read_config, record_environment,
    save_predictions, sha256_file, shuffled_batches, write_json,
)

CFG = read_config(CONFIG_FILE)
random.seed(42)
np.random.seed(42)
STAGE = CFG["stage"]
EXPERIMENT_DIR = Path(CFG["output_root"]) / CFG["experiment_id"]
STAGE_DIR = EXPERIMENT_DIR / STAGE
print("Stage:", STAGE, "| package:", CFG["package_dir"])


Stage: full | package: /home/hembegur/Documents/Swin/cos30049_Inno_Proj/g3b_primary_v1


## 3. A tiny example you can understand by inspection
The first three points are close to one another; the last three form a different group.
Only the coordinates are supplied. The returned integers identify the groups.
The integers may be reversed on another fit and still describe the same partition.


In [6]:
toy_X = np.array([[0, 0], [0, 1], [1, 0], [9, 9], [9, 10], [10, 9]], dtype=float)
with threadpool_limits(limits=CFG["threads"]):
    toy_model = KMeans(n_clusters=2, n_init=10, random_state=42).fit(toy_X)
display(pd.DataFrame({"feature_1": toy_X[:, 0], "feature_2": toy_X[:, 1],
                      "cluster_id": toy_model.labels_}))
print("Centers:\n", toy_model.cluster_centers_)


,feature_1,feature_2,cluster_id
0,0.0,0.0,0
1,0.0,1.0,0
2,1.0,0.0,0
3,9.0,9.0,1
4,9.0,10.0,1
5,10.0,9.0,1


Centers:
 [[0.33333333 0.33333333]
 [9.33333333 9.33333333]]


## 4. Verify and open the real package
Before the first project run, execute G3B's `verify_primary_package.py` as shown in the guide.
This is required to establish evidence on your machine; the PDF says READY_FOR_TEAM, not TEAM_VERIFIED.

`PrimaryPackage(..., verify=True)` also validates the package. Its `load` method returns file-backed
NumPy arrays (**memmaps**): a way to read pieces from disk without materializing the whole matrix
in Python RAM. Never wrap the entire matrix in a pandas DataFrame.

We check the exact manifest checksum supplied in the PDF, the two expected shapes, and each
metadata file's SHA-256 population fingerprint. We use only the loader API shown in the handoff;
we do not guess the internal JSON schema. If a check fails, inspect the actual package with G3B.


In [2]:
package, IDENTITY = open_package(CFG)  # Hash verification may take several minutes.
display(pd.DataFrame({split: info["shape"] for split, info in IDENTITY["partitions"].items()},
                      index=["rows", "features"]).T)
print("Manifest SHA-256:", IDENTITY["manifest_sha256"])

if (STAGE_DIR / "run_index.json").exists():
    raise FileExistsError("This stage is complete. Use replay_predictions.py, or a NEW experiment_id for retraining.")
STAGE_DIR.mkdir(parents=True, exist_ok=True)
write_json(STAGE_DIR / "configuration.json", CFG)
write_json(STAGE_DIR / "package_identity.json", IDENTITY)
record_environment(STAGE_DIR)
manifest_path = Path(CFG["package_dir"]) / IDENTITY["manifest_relative_path"]
shutil.copy2(manifest_path, STAGE_DIR / "source_manifest.json")


,rows,features
TRAIN,4944968,115
VALIDATION,1056372,115


Manifest SHA-256: 6ab99e86397242e219bfa1f8490d8e03b169ec87a092af80cb3e7aef1e61147b


PosixPath('/home/hembegur/Documents/Swin/cos30049_Inno_Proj/G3B_Clustering_Notebook_Kit/outputs/g3b_clustering_001/full/source_manifest.json')

## 5. Features, scaling, and the resource budget
Use **all 115 numeric predictors in their canonical order**. Labels, source identity, row numbers,
device names and taxonomy strings are not added to X. Feature semantics/names must be read from
the package's copied contracts; the PDF alone does not list them.

For standardized feature j, \(z_j=(x_j-\mu_j)/s_j\).
Example: if byte counts are in thousands while another feature is near 0.1, raw Euclidean distance
can be dominated by byte counts. Scaling changes that geometry.

G3B already fit StandardScaler on **exactly TRAIN**. We reuse `chunks(..., scaled=True)`.
We do not fit another scaler, refit on VALIDATION, or scale twice.
`g3b_standard` is the predeclared primary analysis. `raw` is a sensitivity experiment.
Both use identical rows. StandardScaler remains sensitive to outliers, and correlated predictors
can give related behavior extra weight. These are limitations to discuss, not permission to alter G3B silently.

100,000 × 115 float64 values occupy about **87.7 MiB**; one 4,096-row batch is about **3.6 MiB**.
The eight-batch local shuffle buffer is about 28.8 MiB, plus temporary copies and library overhead.
This bounds feature buffers, not total process memory; disk page cache and fitting need extra RAM.
Samples/caches are saved to disk so both stages and all K values use the same observations.


In [3]:
CACHE = prepare_cache(package, CFG, IDENTITY, EXPERIMENT_DIR)
train_indices = np.load(CACHE / "TRAIN_indices.npy", mmap_mode="r")
validation_indices = np.load(CACHE / "VALIDATION_indices.npy", mmap_mode="r")
print("Pilot TRAIN rows:", len(train_indices))
print("VALIDATION feature evaluation rows:", len(validation_indices))
print("Fixed samples:", CACHE)
print("No labels were used to select these uniform samples.")


Pilot TRAIN rows: 100000
VALIDATION feature evaluation rows: 10000
Fixed samples: /home/hembegur/Documents/Swin/cos30049_Inno_Proj/G3B_Clustering_Notebook_Kit/outputs/g3b_clustering_001/cache
No labels were used to select these uniform samples.


## 6. Why try K = 2, 4, 8, 12?
K=2 checks a coarse two-group structure. K=4 and K=8 allow several traffic regimes.
K=12 explores finer partitions. These are modest, predeclared probes; they are **not** assumed
to equal the number of attack classes or devices. Larger K also costs more computation.

The pilot trains eight candidates: 4 K values × 2 scaling choices × seed 42.
Notebook 02 chooses a finalist **within each scaling**, using only internal geometry metrics.
The full stage then trains those two finalists on all TRAIN rows.
The raw baseline cannot replace the primary standardized analysis simply because its
silhouette is larger: scaling changes what the distances mean.

The handoff's shared macro-F1 / FPR / threshold=0.5 contract belongs to classification.
This kit proposes a **separate versioned unsupervised protocol**. Review it with the team before
claiming acceptance under the team's experiment policy; no classifier schema is invented here.


In [4]:
if STAGE == "pilot":
    SPECS = [{"scaling": scale, "k": int(k), "seed": int(seed)}
             for scale, k, seed in itertools.product(CFG["scalings"], CFG["k_values"], CFG["seeds"])]
else:
    plan_file = EXPERIMENT_DIR / "pilot" / "full_plan.json"
    plan = json.loads(plan_file.read_text(encoding="utf-8"))
    if plan["identity"] != IDENTITY or plan["comparison_policy"] != comparison_policy(CFG):
        raise ValueError("Pilot plan population/configuration differs; create a new experiment version")
    SPECS = plan["finalists"]
    # Archive the exact selection input for this full run.
    write_json(STAGE_DIR / "input_full_plan.json", plan)

display(pd.DataFrame(SPECS))


,scaling,k,seed
0,g3b_standard,8,42
1,raw,4,42


## 7. Training: understand the two branches before executing
**Pilot:** `MiniBatchKMeans.fit(X_sample)` repeatedly samples small batches from the uniform
TRAIN sample. `n_init=5` tests five initializations and runs one MiniBatch optimization using
the best initialization; it does not mean five independently trained models.

**Full:** `kmeans_plusplus` initializes centers using a globally uniform TRAIN sample.
We then call `partial_fit(X_batch)` for every batch, for two explicit passes (**epochs**).
`partial_fit` processes the supplied batch; `max_iter` does not automatically repeat the full
stream. We therefore write the epoch loop ourselves. We use `n_init=1` with explicit centers.

The buffer shuffles rows locally, up to eight batches at a time. It is **not a global shuffle**.
Source-ordered streams may still bias the updates. Uniform initialization helps, but does not
prove order independence or convergence. `reassignment_ratio=0` disables recycling low-count
centers during a source-ordered stream; empty or poorly supported centers remain visible in diagnostics.
Two epochs are an explicit resource budget; for a
stronger experiment, predeclare more epochs / seeds in a NEW experiment version and compare
internal validation metrics. Do not modify the stream after inspecting security labels.

Notice that neither `.fit(...)` nor `.partial_fit(...)` receives `y`.


In [5]:
run_index = []
with threadpool_limits(limits=CFG["threads"]):
    for spec in SPECS:
        scaling, k, seed = spec["scaling"], spec["k"], spec["seed"]
        run_id = f"{scaling}_k{k}_seed{seed}"
        run_dir = STAGE_DIR / run_id
        run_dir.mkdir(exist_ok=False)  # Preserve earlier evidence if a cell is accidentally rerun.
        started = time.perf_counter()
        log = (run_dir / "training.jsonl").open("w", encoding="utf-8", buffering=1)
        try:
            def log_event(event, **values):
                record = {"time_utc": datetime.now(timezone.utc).isoformat(), "event": event, **values}
                log.write(json.dumps(record, allow_nan=False) + "\n")

            X_pilot = np.load(CACHE / f"TRAIN_{scaling}.npy", mmap_mode="r")
            if STAGE == "pilot":
                model = MiniBatchKMeans(
                    n_clusters=k, random_state=seed, batch_size=CFG["batch_size"],
                    init="k-means++", n_init=CFG["pilot_n_init"],
                    max_iter=CFG["pilot_max_iter"], max_no_improvement=20,
                    reassignment_ratio=CFG["reassignment_ratio"], compute_labels=False,
                )
                model.fit(X_pilot)  # ONLY FEATURES. This is the declared pilot sample.
                fit_rows = len(X_pilot)
                log_event("pilot_fit_complete", unique_fit_rows=fit_rows,
                          minibatch_updates=int(model.n_steps_))
            else:
                init_rng = np.random.default_rng(seed)
                init_positions = init_rng.choice(len(X_pilot), min(len(X_pilot), CFG["initialization_rows"]),
                                                replace=False)
                initial_centers, _ = kmeans_plusplus(X_pilot[init_positions], n_clusters=k, random_state=seed)
                model = MiniBatchKMeans(
                    n_clusters=k, init=initial_centers, n_init=1, random_state=seed,
                    batch_size=CFG["batch_size"], compute_labels=False,
                    reassignment_ratio=CFG["reassignment_ratio"],
                )
                fit_rows = IDENTITY["partitions"]["TRAIN"]["shape"][0]
                for epoch in range(CFG["full_epochs"]):
                    seen = 0
                    for X_batch in shuffled_batches(package, scaling, CFG["batch_size"],
                                                     CFG["shuffle_buffer_batches"], seed + epoch):
                        model.partial_fit(X_batch)  # ONLY FEATURES; no ground-truth labels.
                        seen += len(X_batch)
                    if seen != fit_rows:
                        raise AssertionError("Did not train on exactly the full TRAIN population")
                    log_event("epoch_complete", epoch=epoch + 1, rows=seen,
                              elapsed_seconds=time.perf_counter() - started)
                    print(run_id, "epoch", epoch + 1, "rows", f"{seen:,}", flush=True)
            fit_seconds = time.perf_counter() - started

            # A bare estimator expects ALREADY transformed features. Keep this requirement explicit.
            bundle = {"estimator": model, "scaling": scaling, "identity": IDENTITY,
                      "feature_policy": "all 115 canonical G3B predictors, original order, float64",
                      "transform": "PrimaryPackage.chunks(scaled=True)" if scaling == "g3b_standard"
                                   else "PrimaryPackage.chunks(scaled=False)",
                      "config": CFG, "spec": spec}
            joblib.dump(bundle, run_dir / "model.joblib")
            write_json(run_dir / "configuration.json", {**spec, "stage": STAGE,
                       "unique_fit_rows": fit_rows, "fit_seconds": fit_seconds,
                       "full_epochs": CFG["full_epochs"] if STAGE == "full" else None,
                       "estimator_parameters": {key: value for key, value in model.get_params().items()
                                                if key != "init"},
                       "initialization": "k-means++ on sample"})
            np.save(run_dir / "cluster_centers.npy", model.cluster_centers_)
            X_validation_sample = np.load(CACHE / f"VALIDATION_{scaling}.npy", mmap_mode="r")
            sample_labels = model.predict(X_validation_sample).astype(np.int16)
            np.save(run_dir / "VALIDATION_sample_clusters.npy", sample_labels)
            coverage = {}
            if STAGE == "pilot":
                np.save(run_dir / "TRAIN_sample_clusters.npy", model.predict(X_pilot).astype(np.int16))
            else:
                # Partial-fit labels are not final assignments. Re-predict every row AFTER training.
                for split in ("TRAIN", "VALIDATION"):
                    coverage[split] = save_predictions(
                        package, model, scaling, split, IDENTITY["partitions"][split]["shape"][0],
                        run_dir / f"{split}_clusters.npy", CFG["batch_size"])
                full_val = np.load(run_dir / "VALIDATION_clusters.npy", mmap_mode="r")
                np.testing.assert_array_equal(full_val[validation_indices], sample_labels)
                write_json(run_dir / "prediction_coverage.json", coverage)
            reloaded = joblib.load(run_dir / "model.joblib")
            np.testing.assert_array_equal(model.predict(X_validation_sample[:64]),
                                          reloaded["estimator"].predict(X_validation_sample[:64]))
            log_event("saved_and_reload_verified", fit_seconds=fit_seconds,
                      total_seconds=time.perf_counter() - started, coverage=coverage)
            run_index.append({"run_id": run_id, **spec, "stage": STAGE,
                              "unique_fit_rows": fit_rows, "fit_seconds": fit_seconds,
                              "model_sha256": sha256_file(run_dir / "model.joblib")})
            print("Saved:", run_id, "| fit seconds:", round(fit_seconds, 2), flush=True)
        except Exception as error:
            write_json(run_dir / "failure.json", {"type": type(error).__name__, "message": str(error)})
            raise  # A failed run is visible, never presented as successful evidence.
        finally:
            log.close()

write_json(STAGE_DIR / "run_index.json", run_index)
display(pd.DataFrame(run_index))


g3b_standard_k8_seed42 epoch 1 rows 4,944,968
g3b_standard_k8_seed42 epoch 2 rows 4,944,968
Saved: g3b_standard_k8_seed42 | fit seconds: 17.83
raw_k4_seed42 epoch 1 rows 4,944,968
raw_k4_seed42 epoch 2 rows 4,944,968
Saved: raw_k4_seed42 | fit seconds: 12.56


,run_id,scaling,k,seed,stage,unique_fit_rows,fit_seconds,model_sha256
0,g3b_standard_k8_seed42,g3b_standard,8,42,full,4944968,17.827990,b35f710aa449ec4a3c0c9009dc1325b6af90849520b7cc...
1,raw_k4_seed42,raw,4,42,full,4944968,12.557584,40590fc5f997294c11a565e6a672c108df1606e78efd33...


## 8. What did you save, and what happens next?
Each run has a fitted `model.joblib`, final centroids, configuration and a timestamped JSONL log.
The stage also records package/population fingerprints, the manifest, runtime versions, and a pip lock.
The cache records the exact sampled row indices. Pilot prediction arrays refer to those indices.
Full prediction arrays contain **every row** in their original partition order; the original
metadata files identify the source rows. Original metadata and labels are not concatenated into features.

The saved joblib bundle contains the estimator and its preprocessing **contract**, not a newly fitted
scaler. Replay intentionally requires the complete, same verified G3B package and its scaler JSON.
It is not yet an inference API for arbitrary packet captures. Load only your own/trusted joblib files.

Now run notebook **02** in the same stage. A pilot run is development evidence, not final full-population
training evidence. Full training and evaluation are still needed to complete the assignment.


## Official references
- [MiniBatchKMeans: fit, partial_fit, n_init](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.MiniBatchKMeans.html)
- [K-Means and its assumptions](https://scikit-learn.org/stable/modules/clustering.html#k-means)
- [StandardScaler and outlier sensitivity](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [Model persistence](https://scikit-learn.org/stable/model_persistence.html)

Input facts (row counts, checksum, feature count, scaler policy and loader API) come from the supplied
`PRIMARY_DATA_HANDOFF.md.pdf`, dated 30 October 2026. Its package and source code were not attached to
this request. Read `VALIDATION_NOTES.md` for exactly what could be tested here.
